# Overview

This notebook is used to manually test the tools before refactoring the code to work with `LangGraph` in the `tools/custom_tools` folder.

In [0]:
%pip install python-dotenv twilio


In [0]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  

MAILGUN_API_URL = os.environ.get("MAILGUN_API_URL")
MAILGUN_API_KEY = os.environ.get("MAILGUN_API_KEY")
SENDER          = os.getenv("SENDER")
RECIPIENT       = os.getenv("RECIPIENT")
SMS_ACCOUNT_SID = os.getenv("SMS_ACCOUNT_SID")
SMS_AUTH_TOKEN  = os.getenv("SMS_AUTH_TOKEN")
SMS_ACCOUNT_ID  = os.getenv("SMS_ACCOUNT_ID")             # optional: AC... (needed when SMS_ACCOUNT_SID is an SK... API key)
SMS_MESSAGING_SERVICE_SID = os.getenv("SMS_MESSAGING_SERVICE_SID")  # optional: MG...

def loaded(val):
    return "✅ loaded" if val else "❌ not loaded"

print(f" - MAILGUN_API_URL: {loaded(MAILGUN_API_URL)}")
print(f" - MAILGUN_API_KEY: {loaded(MAILGUN_API_KEY)}")
print(f" - SENDER: {loaded(SENDER)}")
print(f" - RECIPIENT: {loaded(RECIPIENT)}")
print(f" - SMS_ACCOUNT_SID: {loaded(SMS_ACCOUNT_SID)}")
print(f" - SMS_AUTH_TOKEN: {loaded(SMS_AUTH_TOKEN)}")
print(f" - SMS_ACCOUNT_ID (optional): {loaded(SMS_ACCOUNT_ID)}")
print(f" - SMS_MESSAGING_SERVICE_SID (optional): {loaded(SMS_MESSAGING_SERVICE_SID)}")


## Test Twilio

In [ ]:
import time
from twilio.rest import Client

# Auth, per https://www.twilio.com/docs/messaging/services:
#   - API key:  Client(api_key_sid "SK...", api_key_secret, account_sid "AC...")
#   - Or local: Client(account_sid "AC...", auth_token)
# SMS_ACCOUNT_SID holds either the SK... key or the AC... account SID.
if SMS_ACCOUNT_SID.startswith("SK") and SMS_ACCOUNT_ID:
    client = Client(SMS_ACCOUNT_SID, SMS_AUTH_TOKEN, SMS_ACCOUNT_ID)
else:
    client = Client(SMS_ACCOUNT_SID, SMS_AUTH_TOKEN)

FROM_NUMBER = "+18773559479"   # same number used by tools/custom_tools/tool_send_sms.py
TO_NUMBER   = "+13019085817"

# Use the Messaging Service sender pool if configured, otherwise send from FROM_NUMBER
sender = ({"messaging_service_sid": SMS_MESSAGING_SERVICE_SID}
          if SMS_MESSAGING_SERVICE_SID else {"from_": FROM_NUMBER})
print(f"Sending via: {next(iter(sender))}")

message = client.messages.create(
    body="WeatherWise agent: Twilio SMS test from tests/manually_test_tools.ipynb",
    to=TO_NUMBER,
    **sender,
)
print(f"Queued: SID={message.sid} status={message.status}")

# Poll for a final delivery status
for _ in range(10):
    m = client.messages(message.sid).fetch()
    if m.status in ("delivered", "undelivered", "failed"):
        break
    time.sleep(2)
print(f"Final:  status={m.status} error_code={m.error_code} error_message={m.error_message}")


## Test Mailgun

In [0]:
import requests

def send_simple_message():
    response = requests.post(
        MAILGUN_API_URL,
        auth=("api", MAILGUN_API_KEY),
        data={
            "from": "AI AGENT <robert@datakafe.com>",
            "to": "robert.leach@databricks.com",
            "subject": "Hello robert",
            "text": "This is an email test for my agent!"
        }
    )
    return response

# Invoke the function and print the response object
response = send_simple_message()
display(response)